In [36]:
"""
The purpose of this Jupyter notebook is to refine the test set
intensities via multiplication by the probabilities outputted by the PU
learning model.

This refinement step is performed for the PPI-only model, the model
trained on row-permuted PPI features without phenotypic features as well
as the model trained on column-permuted PPI features without phenotypic
features.
"""

'\nThe purpose of this Jupyter notebook is to refine the test set\nintensities via multiplication by the probabilities outputted by the PU\nlearning model.\n\nThis refinement step is performed for the PPI-only model, the model\ntrained on row-permuted PPI features without phenotypic features as well\nas the model trained on column-permuted PPI features without phenotypic\nfeatures.\n'

In [37]:
import os

import pandas as pd

# Load the Mean Intensities

In [38]:
mean_ints_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/screen_subset_mean_"
    "features.tsv"
)

mean_ints_df = pd.read_csv(
    mean_ints_path,
    sep="\t"
)

# Remove duplicate rows
mean_ints_df.drop_duplicates(
    "Name",
    inplace=True,
    ignore_index=True
)

# Defining a Function for Intensity Refinement

In [39]:
# For the sake of convenience, a function is defined bundling the
# individual steps of intensity refinement
def intensity_refinement(
        prob_df, int_df, early_int_col_name, late_int_col_name,
        output_dir, output_file_info
):
    """
    Performs intensity refinement, i.e. refines intensities by
    multiplying them by predicted probabilities.

    The intensity refinement is performed for early as well as late
    intensities.

    Parameters
    ----------
    prob_df: Pandas DataFrame
        A Pandas DataFrame storing the predicted probabilities. It must
        comprise two columns, the first of which bears the name `gene`
        and the second of which bears the name `probability`.
    int_df: Pandas DataFrame
        A Pandas DataFrame storing the intensities. It must contain a
        `Name` column matching the gene names in the `gene` column of
        `prob_df`. The column containing the early intensities to be
        refined is denoted by the `early_int_col_name` parameter,
        whereas the column containing the late intensities to be refined
        is denoted by the `late_int_col_name` parameter.
    output_dir: str
        A string denoting the output directory to store the refined
        intensities in.
    output_file_info: str
        A string denoting information about the output file storing the
        refined intensities. The string is inserted into the following
        template:
        "refined_{early/late}_intensities_{output_file_info}.tsv"
    
    Returns
    -------
    None
    """
    # As a first step, filter the intensity DataFrame to retain only
    # genes also present in the probability DataFrame
    int_df = int_df[int_df["Name"].isin(prob_df["gene"])]

    # As a second step, create a Series mapping gene names to the
    # predicted probabilities
    prob_map = prob_df.set_index("gene")["probability"]

    # Now, multiply the intensities by the probabilities so as to obtain
    # the refined intensities
    refined_early_ints_df = int_df[["Name"]].rename(
        columns={"Name": "gene"}
    ).copy()
    refined_early_ints_df["probability"] = (
        int_df[early_int_col_name]
        *
        int_df["Name"].map(prob_map)
    )

    refined_late_ints_df = int_df[["Name"]].rename(
        columns={"Name": "gene"}
    ).copy()
    refined_late_ints_df["probability"] = (
        int_df[late_int_col_name]
        *
        int_df["Name"].map(prob_map)
    )
    
    # As a last step, save the two DataFrames storing the refined
    # intensities to disk
    refined_early_ints_df.to_csv(
        os.path.join(
            output_dir,
            f"refined_early_intensities_{output_file_info}.tsv"
        ),
        sep="\t",
        index=False
    )

    refined_late_ints_df.to_csv(
        os.path.join(
            output_dir,
            f"refined_late_intensities_{output_file_info}.tsv"
        ),
        sep="\t",
        index=False
    )

# Perform the Intensity Refinement for the PPI-Only Model

In [40]:
ppi_only_dir = "inference_results_only_PPI_features"
ppi_only_preds_file_template = (
    "all_hf_test_set_predictions_only_PPIs_seed_{i}.tsv"
)

for i in range(42, 47):
    current_preds_path = os.path.join(
        ppi_only_dir,
        ppi_only_preds_file_template.format(i=i)
    )
    current_preds_df = pd.read_csv(
        current_preds_path,
        sep="\t"
    )
    intensity_refinement(
        current_preds_df,
        mean_ints_df,
        "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
        "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
        ppi_only_dir,
        f"test_set_PPI-only_model_seed_{i}"
    )

# Perform the Intensity Refinement for the Row-Permuted PPI-Feature Model Trained without Phenotypic Features

In [41]:
row_permuted_ppi_wo_phenotype_dir = (
    "inference_results_row_permutation_of_PPI_without_phenotype"
)
row_permuted_ppi_wo_phenotype_preds_file_template = (
    "all_hf_test_set_predictions_row_permutation_of_PPI_vecs_without_"
    "phenotype_seed_{i}.tsv"
)

for i in range(42, 47):
    current_preds_path = os.path.join(
        row_permuted_ppi_wo_phenotype_dir,
        row_permuted_ppi_wo_phenotype_preds_file_template.format(i=i)
    )
    current_preds_df = pd.read_csv(
        current_preds_path,
        sep="\t"
    )
    intensity_refinement(
        current_preds_df,
        mean_ints_df,
        "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
        "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
        row_permuted_ppi_wo_phenotype_dir,
        "test_set_row-permuted_PPI-feature_model_without_phenotypic_"
        f"features_seed_{i}"
    )

# Perform the Intensity Refinement for the Column-Permuted PPI-Feature Model Trained without Phenotypic Features

In [42]:
col_permuted_ppi_wo_phenotype_dir = (
    "inference_results_col_permutation_of_PPI_without_phenotype"
)
col_permuted_ppi_wo_phenotype_preds_file_template = (
    "all_hf_test_set_predictions_col_permutation_of_PPI_vecs_without_"
    "phenotype_seed_{i}.tsv"
)

for i in range(42, 47):
    current_preds_path = os.path.join(
        col_permuted_ppi_wo_phenotype_dir,
        col_permuted_ppi_wo_phenotype_preds_file_template.format(i=i)
    )
    current_preds_df = pd.read_csv(
        current_preds_path,
        sep="\t"
    )
    intensity_refinement(
        current_preds_df,
        mean_ints_df,
        "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore",
        "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore",
        col_permuted_ppi_wo_phenotype_dir,
        "test_set_column-permuted_PPI-feature_model_without_phenotypic_"
        f"features_seed_{i}"
    )